<a href="https://colab.research.google.com/github/eliasdengo/Python-practice-from-beginning-to-advance/blob/Eliasrepos/Recognize_Handwritten_Digits.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np
from PIL import Image


In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# If one-hot encoding is desired:
y_train_one_hot = tf.keras.utils.to_categorical(y_train, num_classes=10)
y_test_one_hot = tf.keras.utils.to_categorical(y_test, num_classes=10)

print("x_train shape:", x_train.shape)
print("y_train_one_hot shape:", y_train_one_hot.shape)

x_train shape: (60000, 28, 28)
y_train_one_hot shape: (60000, 10)


In [ ]:
n_input = 784 # input layer (28x28 pixels)
n_hidden1 = 512 # 1st hidden layer
n_hidden2 = 256 # 2nd hidden layer
n_hidden3 = 128 # 3rd hidden layer
n_output = 10 # output layer (0-9 digits)

In [ ]:
learning_rate = 1e-4
n_iterations = 1000
batch_size = 128
dropout = 0.5

In [ ]:
X_input = tf.keras.Input(shape=(n_input,), name='input_features')
Y_target = tf.keras.Input(shape=(n_output,), name='target_labels')
# Dropout will be handled by a tf.keras.layers.Dropout layer within the model architecture.

In [ ]:
weights = {
'w1': tf.Variable(tf.random.truncated_normal([n_input, n_hidden1],
stddev=0.1)),
'w2': tf.Variable(tf.random.truncated_normal([n_hidden1, n_hidden2],
stddev=0.1)),
'w3': tf.Variable(tf.random.truncated_normal([n_hidden2, n_hidden3],
stddev=0.1)),
'out': tf.Variable(tf.random.truncated_normal([n_hidden3, n_output],
stddev=0.1)),
}

In [ ]:
biases = {
'b1': tf.Variable(tf.constant(0.1, shape=[n_hidden1])),
'b2': tf.Variable(tf.constant(0.1, shape=[n_hidden2])),
'b3': tf.Variable(tf.constant(0.1, shape=[n_hidden3])),
'out': tf.Variable(tf.constant(0.1, shape=[n_output]))
}

In [ ]:
# Layer 1
layer_1_matmul = tf.keras.layers.Lambda(lambda x: tf.matmul(x, weights['w1']), name='layer1_matmul')(X_input)
layer_1 = tf.keras.layers.Lambda(lambda x: tf.add(x, biases['b1']), name='layer1_add')(layer_1_matmul)

# Layer 2
layer_2_matmul = tf.keras.layers.Lambda(lambda x: tf.matmul(x, weights['w2']), name='layer2_matmul')(layer_1)
layer_2 = tf.keras.layers.Lambda(lambda x: tf.add(x, biases['b2']), name='layer2_add')(layer_2_matmul)

# Layer 3
layer_3_matmul = tf.keras.layers.Lambda(lambda x: tf.matmul(x, weights['w3']), name='layer3_matmul')(layer_2)
layer_3 = tf.keras.layers.Lambda(lambda x: tf.add(x, biases['b3']), name='layer3_add')(layer_3_matmul)

# Dropout Layer
layer_drop = tf.keras.layers.Dropout(rate=dropout)(layer_3)

# Output Layer
output_layer_matmul = tf.keras.layers.Lambda(lambda x: tf.matmul(x, weights['out']), name='output_matmul')(layer_drop)
output_layer = tf.keras.layers.Lambda(lambda x: tf.add(x, biases['out']), name='output_add')(output_layer_matmul)

In [ ]:
cross_entropy = tf.keras.layers.Lambda(
    lambda args: tf.reduce_mean(
        tf.nn.softmax_cross_entropy_with_logits(labels=args[0], logits=args[1])
    )
)([Y_target, output_layer])
# The training step will be handled by compiling and fitting a tf.keras.Model.
# tf.train.AdamOptimizer is deprecated in TF2.x.

In [ ]:
correct_pred = tf.keras.layers.Lambda(
    lambda args: tf.equal(tf.argmax(args[0], 1), tf.argmax(args[1], 1))
)([output_layer, Y_target])
accuracy = tf.keras.layers.Lambda(lambda x: tf.reduce_mean(tf.cast(x, tf.float32)))(correct_pred)

In [ ]:
# This TensorFlow 1.x evaluation code is no longer needed.
# The model will be evaluated using model.evaluate() after training.
print("Accuracy:", accuracy)

Accuracy: <KerasTensor shape=(), dtype=float32, sparse=False, ragged=False, name=keras_tensor_269>


In [ ]:
# Select an image from the test set for prediction
# For example, let's use the first image from x_test
# Remember that x_test_flat was already prepared during model compilation

# Assuming x_test_flat and model are available from previous steps
if 'x_test_flat' in locals() and 'model' in locals():
    sample_image_flat = x_test_flat[0:1] # Get the first image, keeping its dimension for prediction
    sample_label = y_test[0]

    # Make a prediction using the trained Keras model
    prediction_probabilities = tf.keras.Model.predict(sample_image_flat)
    predicted_class = tf.argmax(prediction_probabilities, axis=1).numpy()

    print(f"Original Label for test image: {sample_label}")
    print(f"Prediction for test image: {np.squeeze(predicted_class)}")
else:
    print("Please ensure 'x_test_flat' and 'model' are defined by running previous cells.")

Please ensure 'x_test_flat' and 'model' are defined by running previous cells.
